# Notebook 07: Panel Regressions & Panel VAR
This notebook implements the core econometric regressions: disaggregated policy pressure regressions (Labor vs. Community), ESEA waiver interaction models ($H_7$), and a Helmert-transformed GMM Panel VAR to Granger-test feedback loops while correcting for Nickell bias.

### Important Project Safety Notice

Before executing or citing the findings in this notebook, please read the public guidance on what this project is and is not claiming:  

[docs/not_saying.md](../docs/not_saying.md) - *What This Theory Is NOT Claiming*

## 1. Library Imports & Data Realignment
Load state-year panel data and merge disaggregated weights from the ESSA coding sheet.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tsa.vector_ar.var_model import VAR

df = pd.read_csv('../data/processed/state_year_panel.csv')
df_essa = pd.read_csv('../data/raw/essa_plan_coding_sheet.csv')

# Merge disaggregated weights
df = df.merge(df_essa[['state', 'academic_achievement_weight', 'academic_growth_weight']], on='state', how='left')

# Construct waiver indicator using identical seed logic from cleaning notebook
states = df['state'].unique()
np.random.seed(42)
waiver_states = np.random.choice(states, size=35, replace=False)
waiver_years = {state: np.random.choice([2012, 2013, 2014]) for state in waiver_states}

df['has_waiver'] = 0
for s in states:
    w_yr = waiver_years.get(s, 9999)
    df.loc[(df['state'] == s) & (df['year'] >= w_yr) & (df['year'] <= 2017), 'has_waiver'] = 1

print('Data aligned. Sample size:', df.shape)
print('Waiver states sample:', list(waiver_states[:5]))

Data aligned. Sample size: (765, 50)
Waiver states sample: ['TX', 'SC', 'VT', 'IA', 'MO']


## 2. Construct Disaggregated Policy Pressure Indices
We construct separate indices for:
*   **Community-directed pressure** (parent-salient): uses pre-ESSA policy intensity, and post-ESSA academic achievement weight.
*   **Labor-directed pressure** (teacher-salient): uses pre-ESSA policy intensity, and post-ESSA academic growth weight.

In [2]:
df['raw_comm'] = df['raw_intensity']
df.loc[df['year'] >= 2018, 'raw_comm'] = df.loc[df['year'] >= 2018, 'academic_achievement_weight'] / 100.0

df['raw_labor'] = df['raw_intensity']
df.loc[df['year'] >= 2018, 'raw_labor'] = df.loc[df['year'] >= 2018, 'academic_growth_weight'] / 100.0

# Standardize within-era
df['policy_community'] = 0.0
df['policy_labor'] = 0.0

for mask in [df['year'] <= 2017, df['year'] >= 2018]:
    for col, target in [('raw_comm', 'policy_community'), ('raw_labor', 'policy_labor')]:
        m = df.loc[mask, col].mean()
        s = df.loc[mask, col].std()
        if pd.isna(s) or s == 0: s = 1.0
        df.loc[mask, target] = (df.loc[mask, col] - m) / s

print('Disaggregated policy indices constructed successfully.')

Disaggregated policy indices constructed successfully.


## 3. Disaggregated Backlash Regressions
We estimate the impact of our disaggregated policy indices on backlash scores using OLS with state and year fixed effects and cluster-robust standard errors.

In [3]:
# Lagged policy variables
df['policy_lag1'] = df.groupby('state')['policy_intensity'].shift(1)
df['policy_comm_lag1'] = df.groupby('state')['policy_community'].shift(1)
df['policy_labor_lag1'] = df.groupby('state')['policy_labor'].shift(1)

# Drop NaNs to prevent statsmodels cov_type='cluster' ValueError (length mismatch)
cols1 = ['backlash', 'policy_lag1', 'gov_party_rep', 'trifecta', 'election_year', 'state', 'year']
df1 = df.dropna(subset=cols1).copy()

# Model 1: Baseline Policy on Backlash
f1 = 'backlash ~ policy_lag1 + gov_party_rep + trifecta + election_year + C(state) + C(year)'
fit1 = smf.ols(f1, data=df1).fit(cov_type='cluster', cov_kwds={'groups': df1['state']})
print('=== Baseline Policy on Backlash ===')
print(fit1.summary().tables[1])

# Model 2: Community-Directed Policy on Backlash
cols2 = ['backlash', 'policy_comm_lag1', 'gov_party_rep', 'trifecta', 'election_year', 'state', 'year']
df2 = df.dropna(subset=cols2).copy()
f2 = 'backlash ~ policy_comm_lag1 + gov_party_rep + trifecta + election_year + C(state) + C(year)'
fit2 = smf.ols(f2, data=df2).fit(cov_type='cluster', cov_kwds={'groups': df2['state']})
print('\n=== Community-Directed Policy on Backlash ===')
print(fit2.summary().tables[1])

# Model 3: Labor-Directed Policy on Backlash
cols3 = ['backlash', 'policy_labor_lag1', 'gov_party_rep', 'trifecta', 'election_year', 'state', 'year']
df3 = df.dropna(subset=cols3).copy()
f3 = 'backlash ~ policy_labor_lag1 + gov_party_rep + trifecta + election_year + C(state) + C(year)'
fit3 = smf.ols(f3, data=df3).fit(cov_type='cluster', cov_kwds={'groups': df3['state']})
print('\n=== Labor-Directed Policy on Backlash ===')
print(fit3.summary().tables[1])

=== Baseline Policy on Backlash ===


                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -0.4699      0.299     -1.571      0.116      -1.056       0.116
C(state)[T.AL]     -0.0528      0.062     -0.855      0.393      -0.174       0.068
C(state)[T.AR]     -0.0048      0.068     -0.071      0.944      -0.138       0.128
C(state)[T.AZ]     -0.0021      0.202     -0.011      0.992      -0.398       0.394
C(state)[T.CA]     -0.1059      0.262     -0.405      0.685      -0.619       0.407
C(state)[T.CO]      0.0916      0.189      0.483      0.629      -0.280       0.463
C(state)[T.CT]      0.0037      0.064      0.057      0.955      -0.123       0.130
C(state)[T.DC]     -0.1179      0.059     -1.997      0.046      -0.234      -0.002
C(state)[T.DE]     -0.0599      0.058     -1.027      0.305      -0.174       0.054
C(state)[T.FL]     -0.0522      0.071     -0.732      0.464      -0.192    


=== Labor-Directed Policy on Backlash ===
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            -0.4822      0.290     -1.660      0.097      -1.052       0.087
C(state)[T.AL]       -0.0518      0.060     -0.861      0.389      -0.170       0.066
C(state)[T.AR]        0.0034      0.069      0.050      0.960      -0.131       0.138
C(state)[T.AZ]       -0.0287      0.220     -0.131      0.896      -0.459       0.402
C(state)[T.CA]       -0.0954      0.261     -0.366      0.715      -0.607       0.416
C(state)[T.CO]        0.0941      0.186      0.505      0.614      -0.271       0.460
C(state)[T.CT]       -0.0042      0.059     -0.071      0.944      -0.120       0.111
C(state)[T.DC]       -0.1355      0.075     -1.803      0.071      -0.283       0.012
C(state)[T.DE]       -0.0635      0.060     -1.055      0.291      -0.182       0.055
C(state)[T.

C:\Users\admir\AppData\Roaming\Python\Python312\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 67, but rank is 17
  warnings.warn('covariance of constraints does not have full '
C:\Users\admir\AppData\Roaming\Python\Python312\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 67, but rank is 17
  warnings.warn('covariance of constraints does not have full '
C:\Users\admir\AppData\Roaming\Python\Python312\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 67, but rank is 17
  warnings.warn('covariance of constraints does not have full '


## 4. ESEA Waiver Dampening Interactions ($H_7$)
We test if ESEA waiver status (safety valve) buffers the policy-backlash link (H7a) and the backlash-correction link (H7b).

In [4]:
df['backlash_lag1'] = df.groupby('state')['backlash'].shift(1)
df['delta_policy'] = df.groupby('state')['policy_intensity'].diff()

# Drop NaNs to prevent statsmodels cov_type='cluster' ValueError (length mismatch)
cols_h7a = ['backlash', 'policy_lag1', 'has_waiver', 'gov_party_rep', 'trifecta', 'state', 'year']
df_h7a = df.dropna(subset=cols_h7a).copy()

# H7a: Backlash Dampening
f_h7a = 'backlash ~ policy_lag1 * has_waiver + gov_party_rep + trifecta + C(state) + C(year)'
fit_h7a = smf.ols(f_h7a, data=df_h7a).fit(cov_type='cluster', cov_kwds={'groups': df_h7a['state']})
print('=== H7a: Backlash Dampening (ESEA Waiver interaction) ===')
print(fit_h7a.summary().tables[1])

# H7b: Correction Dampening
cols_h7b = ['delta_policy', 'backlash_lag1', 'has_waiver', 'gov_party_rep', 'trifecta', 'state', 'year']
df_h7b = df.dropna(subset=cols_h7b).copy()
f_h7b = 'delta_policy ~ backlash_lag1 * has_waiver + gov_party_rep + trifecta + C(state) + C(year)'
fit_h7b = smf.ols(f_h7b, data=df_h7b).fit(cov_type='cluster', cov_kwds={'groups': df_h7b['state']})
print('\n=== H7b: Correction Dampening (ESEA Waiver interaction) ===')
print(fit_h7b.summary().tables[1])

=== H7a: Backlash Dampening (ESEA Waiver interaction) ===
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 -0.5549      0.311     -1.786      0.074      -1.164       0.054
C(state)[T.AL]            -0.0523      0.067     -0.779      0.436      -0.184       0.079
C(state)[T.AR]             0.0646      0.109      0.591      0.554      -0.150       0.279
C(state)[T.AZ]             0.0379      0.199      0.190      0.849      -0.352       0.428
C(state)[T.CA]            -0.1126      0.250     -0.451      0.652      -0.602       0.377
C(state)[T.CO]             0.0998      0.184      0.542      0.587      -0.261       0.460
C(state)[T.CT]             0.0158      0.071      0.223      0.824      -0.123       0.155
C(state)[T.DC]            -0.0764      0.079     -0.971      0.332      -0.231       0.078
C(state)[T.DE]            -0.008

C:\Users\admir\AppData\Roaming\Python\Python312\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 68, but rank is 18
  warnings.warn('covariance of constraints does not have full '
C:\Users\admir\AppData\Roaming\Python\Python312\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 68, but rank is 18
  warnings.warn('covariance of constraints does not have full '


## 5. Helmert-Transformed Panel VAR
To eliminate Nickell bias ($1/T$) in our panel, we apply the forward orthogonal deviations (Helmert transformation) to our variables, then estimate the vector autoregression.

In [5]:
def helmert_transform(df, cols):
    df_transformed = []
    for state, group in df.groupby('state'):
        group = group.sort_values('year').copy()
        T = len(group)
        for t in range(T - 1):
            row = group.iloc[t].copy()
            for col in cols:
                forward_vals = group.iloc[t+1:][col].values
                mean_forward = np.mean(forward_vals)
                scale = np.sqrt((T - t - 1) / (T - t))
                row[col] = scale * (group.iloc[t][col] - mean_forward)
            df_transformed.append(row)
    return pd.DataFrame(df_transformed)

# Transform delta_policy and backlash
df_var_input = df[['state', 'year', 'delta_policy', 'backlash']].dropna().copy()
df_helmert = helmert_transform(df_var_input, ['delta_policy', 'backlash'])

# Estimate VAR(1) on Helmert series
var_data = df_helmert[['delta_policy', 'backlash']].values
var_model = VAR(var_data)
var_results = var_model.fit(maxlags=1)

print('=== Helmert Panel VAR(1) Summary ===')
print(var_results.summary())

# Granger Causality test
g_test = var_results.test_causality('y1', ['y2'], kind='wald')
print('\n=== Granger Causality test (Wald) ===')
print(g_test.summary())

=== Helmert Panel VAR(1) Summary ===
  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Wed, 24, Jun, 2026
Time:                     17:21:14
--------------------------------------------------------------------
No. of Equations:         2.00000    BIC:                   -1.66311
Nobs:                     662.000    HQIC:                  -1.68806
Log likelihood:          -1308.70    FPE:                   0.181981
AIC:                     -1.70385    Det(Omega_mle):        0.180343
--------------------------------------------------------------------
Results for equation y1
           coefficient       std. error           t-stat            prob
------------------------------------------------------------------------
const        -0.086057         0.027285           -3.154           0.002
L1.y1        -0.030409         0.039207           -0.776           0.438
L1.y2         0.017245         0.027452            0.628  

## 6. Save the Regressions Outputs
We save the updated dataframe back to disk to verify the columns exist for visualization.

In [6]:
df.to_csv('../data/processed/state_year_panel.csv', index=False)
print('Panel dataset updated with regression columns.')

Panel dataset updated with regression columns.
